# Day 1: เช็กคุณภาพข้อมูลดิบ (Data Check)

เช็กข้อมูล `data/raw/openaq_hourly_raw.csv` (448,609 แถว จาก 3 สถานี) ก่อนนำไปทำความสะอาดจริงใน `02_cleaning.ipynb`

สถานี:
- **225611** Health Promotion Hospital Maptaput (ใกล้นิคม, 4.5 กม.)
- **7846** Field Crop Research Center, Rayong (ใกล้นิคม, 5.8 กม.)
- **717** Pluakdaeng District Health Office (ควบคุม, 33.4 กม.)

In [1]:
import pandas as pd

df = pd.read_csv('../data/raw/openaq_hourly_raw.csv')
print(f'จำนวนแถวทั้งหมด: {len(df)}')
print()
print('ค่า min/max/mean แยกตามพารามิเตอร์')
print(df.groupby('parameter')['value'].agg(['min','max','mean']).round(4))
print()
print('จำนวนแถวที่ค่าติดลบ')
neg = df[df['value'] < 0]
print(f'พบ {len(neg)} แถว')


จำนวนแถวทั้งหมด: 448609

ค่า min/max/mean แยกตามพารามิเตอร์
           min      max     mean
parameter                       
co         0.0   27.100   0.4254
no2        0.0    0.332   0.0086
o3         0.0    0.412   0.0191
pm10       0.0  169.000  43.4644
pm25       1.0  115.000  20.3488
so2        0.0    0.324   0.0028

จำนวนแถวที่ค่าติดลบ
พบ 0 แถว


**สรุป:** ไม่มีค่าติดลบเลย ค่า min/max ของทุกสารอยู่ในช่วงที่สมเหตุสมผล 

## 2. เช็กข้อมูลซ้ำ

เช็กว่ามีแถวที่ "สถานีเดียวกัน + สารเดียวกัน + เวลาเดียวกัน" มากกว่า 1 ครั้งไหม

In [2]:
df['datetime_utc'] = pd.to_datetime(df['datetime_utc'])
dup = df.groupby(['station_id','parameter','datetime_utc']).size()
dup = dup[dup > 1]
n_groups = df.groupby(['station_id','parameter']).ngroups
print(f'พบ {len(dup)} กลุ่มที่ซ้ำ จากทั้งหมด {n_groups} กลุ่ม station+parameter')


พบ 0 กลุ่มที่ซ้ำ จากทั้งหมด 17 กลุ่ม station+parameter


**สรุป:** ไม่มีข้อมูลซ้ำเลยสักแถว (0 จาก 17 กลุ่ม station+parameter)

## 3. เช็กเซนเซอร์ค้าง

หาความยาวที่นานที่สุดของ "ค่าเดิมซ้ำติดกัน" ในแต่ละสถานี/สาร ถ้าค่าเดิมซ้ำนานผิดปกติ แปลว่าเซนเซอร์หยุดทำงานแต่ยังส่งค่าเดิมซ้ำ ๆ

In [3]:
df = df.sort_values(['station_id','parameter','datetime_utc'])
results = []
for (sid, param), g in df.groupby(['station_id','parameter']):
    g = g.sort_values('datetime_utc')
    same_as_prev = g['value'].eq(g['value'].shift())
    run_id = (~same_as_prev).cumsum()
    run_lengths = g.groupby(run_id).size()
    results.append({'station_id': sid, 'parameter': param, 'max_stuck_hours': run_lengths.max()})

res_df = pd.DataFrame(results).sort_values('max_stuck_hours', ascending=False)
print(res_df.to_string(index=False))


 station_id parameter  max_stuck_hours
        717       no2             2510
        717       so2              128
        717        o3              127
     225611       so2              117
     225611       no2               98
       7846       so2               52
     225611      pm25               39
        717      pm25               35
     225611        co               32
       7846        o3               31
     225611      pm10               30
        717      pm10               27
       7846       no2               23
       7846      pm10               21
       7846      pm25               18
     225611        o3               14
       7846        co               11


**พบปัญหา:** NO₂ ของสถานีควบคุม (717) ค่าเดิมซ้ำติดกันนานถึง 2,510 ชั่วโมง (ประมาณ 105 วัน) ซึ่งผิดปกติชัดเจน ดูช่วงนี้ต่อ

In [2]:
g = df[(df.station_id==717) & (df.parameter=='no2')].sort_values('datetime_utc').reset_index(drop=True)
same_as_prev = g['value'].eq(g['value'].shift())
run_id = (~same_as_prev).cumsum()
run_lengths = g.groupby(run_id).size()
worst_run = run_lengths.idxmax()
chunk = g[run_id == worst_run]
print('ค่าที่ค้าง:', chunk['value'].iloc[0])
print('ตั้งแต่:', chunk['datetime_utc'].min())
print('ถึง:', chunk['datetime_utc'].max())
print('จำนวนชั่วโมง:', len(chunk))
print()
print('ภาพรวม NO2 ของสถานีนี้ทั้งหมด')
print('รวมทั้งหมด:', len(g), 'ชั่วโมง')
print('ค่าที่พบบ่อยที่สุด 5 อันดับ:')
print(g['value'].value_counts().head())


ค่าที่ค้าง: 0.0
ตั้งแต่: 2022-09-11T03:00:00Z
ถึง: 2023-01-30T04:00:00Z
จำนวนชั่วโมง: 2510

ภาพรวม NO2 ของสถานีนี้ทั้งหมด
รวมทั้งหมด: 38829 ชั่วโมง
ค่าที่พบบ่อยที่สุด 5 อันดับ:
value
0.000    4132
0.004    3285
0.005    3213
0.003    3146
0.006    2939
Name: count, dtype: int64


**สรุป:** เซนเซอร์ NO₂ ของสถานีควบคุมค้างที่ค่า **0.000** ตั้งแต่ 2022-09-11 ถึง 2023-01-30 (วันสุดท้ายที่สถานีนี้ส่งข้อมูล NO₂ เลย) ซึ่งเป็นไปไม่ได้ที่อากาศจะคงที่ 0 นาน 105 วัน อาจเป็นเพราะเซนเซอร์เสีย

## 4. ความละเอียดของค่าแต่ละสาร

เช็กว่าทำไมจะใช้เกณฑ์ "ซ้ำเกิน N ชั่วโมง" ตัวเดียวกับทุกสารไม่ได้

เพราะสารที่มีความละเอียดของค่าต่ำ (จำนวนค่าที่เป็นไปได้น้อย) จะซ้ำกันเองโดยธรรมชาติบ่อยกว่า

In [7]:
for p, g in df.groupby('parameter'):
    v = g['value'].dropna()
    is_int = (v % 1 == 0).all()
    print(f'{p:6s} หน่วย = {g["units"].iloc[0]:7s} ค่าไม่ซ้ำ ={v.nunique():5d}  ช่วง {v.min()}-{v.max()}  เป็นจำนวนเต็มทั้งหมด={is_int}')


co     หน่วย = ppm     ค่าไม่ซ้ำ =  238  ช่วง 0.0-27.1  เป็นจำนวนเต็มทั้งหมด=False
no2    หน่วย = ppm     ค่าไม่ซ้ำ =   85  ช่วง 0.0-0.332  เป็นจำนวนเต็มทั้งหมด=False
o3     หน่วย = ppm     ค่าไม่ซ้ำ =  154  ช่วง 0.0-0.412  เป็นจำนวนเต็มทั้งหมด=False
pm10   หน่วย = µg/m³   ค่าไม่ซ้ำ =  166  ช่วง 0.0-169.0  เป็นจำนวนเต็มทั้งหมด=True
pm25   หน่วย = µg/m³   ค่าไม่ซ้ำ =  883  ช่วง 1.0-115.0  เป็นจำนวนเต็มทั้งหมด=False
so2    หน่วย = ppm     ค่าไม่ซ้ำ =  136  ช่วง 0.0-0.324  เป็นจำนวนเต็มทั้งหมด=False


**สรุป:** NO₂ มีค่าที่ต่างกันแค่ 85 ค่าในทั้งชุดข้อมูล (เทียบกับ PM2.5 ที่มี 883 ค่า) จึงมีโอกาสซ้ำกันเองตามธรรมชาติสูงกว่ามาก ต้องแยกเกณฑ์ตามสาร ไม่ใช้ N เดียวกันหมด

## 5. เปรียบเทียบความยาวช่วงซ้ำ: ค่า = 0 เทียบกับ ค่าไม่เป็น 0

สมมติฐาน: ค่า 0 ซ้ำกันสั้น ๆ มักเป็น "below detection limit" (อากาศสะอาดจนวัดไม่ได้ ค่าจริง) ส่วนค่าไม่เป็น 0 ซ้ำกันยาว ๆ น่าสงสัยกว่า ต้องแยกเกณฑ์ 2 แบบ

In [8]:
rows=[]
for (sid,p), g in df.groupby(['station_id','parameter']):
    g=g.sort_values('datetime_utc')
    run_id=(~g['value'].eq(g['value'].shift())).cumsum()
    runs=g.groupby(run_id).agg(n=('value','size'), val=('value','first'))
    for is_zero, sub in runs.groupby(runs['val']==0):
        rows.append({'station_id':sid,'parameter':p,'zero_run':is_zero,
                     'n_runs':len(sub),'p50':sub['n'].median(),'p99':sub['n'].quantile(.99),'max':sub['n'].max()})
r=pd.DataFrame(rows)
print('ความยาวช่วงค่าซ้ำ แยก ค่า=0 กับ ค่าไม่เป็น 0')
print(r.to_string(index=False))


ความยาวช่วงค่าซ้ำ แยก ค่า=0 กับ ค่าไม่เป็น 0
 station_id parameter  zero_run  n_runs  p50   p99  max
        717       no2     False   28207  1.0  4.00    9
        717       no2      True     678  2.0 11.23 2510
        717        o3     False   38632  1.0  3.00   31
        717        o3      True    1279  3.0 11.00  127
        717      pm10     False   20072  1.0 10.00   27
        717      pm10      True       2 12.0 22.78   23
        717      pm25     False   20402  1.0 10.00   35
        717       so2     False    9858  1.0 16.00   76
        717       so2      True    3190  4.0 45.00  128
       7846        co     False   15459  1.0  3.00   10
       7846        co      True      40  1.5  9.83   11
       7846       no2     False   13426  1.0  4.00    8
       7846       no2      True     138  2.0 21.63   23
       7846        o3     False   20712  1.0  4.00   10
       7846        o3      True     113  2.0 24.16   31
       7846      pm10     False    7245  2.0 11.00   21
   

**สรุป:** ยืนยันสมมติฐาน ค่า 0 ซ้ำกันบ่อยและยาวกว่าค่าไม่เป็น 0 มากในแทบทุกสถานี/สาร (เช่น SO₂ สถานีควบคุมมีช่วงค่า 0 ซ้ำถึง 3,190 ครั้งแยกกัน ซึ่งเป็นเรื่องปกติของอากาศสะอาด) จึงกำหนดเกณฑ์แยก 2 แบบ:
- ค่า **ไม่เป็น 0** ซ้ำติดกัน **≥ 24 ชั่วโมง** → น่าสงสัย
- ค่า **เป็น 0** ซ้ำติดกัน **≥ 72 ชั่วโมง** → น่าสงสัย (ผ่อนปรนกว่า เพราะ 0 สั้น ๆ เป็นค่าจริงได้บ่อย)

## 6. สรุป: คำนวณ % ข้อมูลที่ถูก flag ตามเกณฑ์ที่กำหนด

วัดผลกระทบก่อนนำไปใช้จริงใน `02_cleaning.ipynb`

In [14]:
ZERO_MIN, NONZERO_MIN = 72, 24
out=[]
for (sid,p), g in df.groupby(['station_id','parameter']):
    g=g.sort_values('datetime_utc').copy()
    run_id=(~g['value'].eq(g['value'].shift())).cumsum()
    n=g.groupby(run_id)['value'].transform('size')
    is_zero=g['value'].eq(0)
    flag=((is_zero)&(n>=ZERO_MIN))|((~is_zero)&(n>=NONZERO_MIN))
    out.append({'station_id':sid,'parameter':p,'total_hours':len(g),
                'flagged':int(flag.sum()),'pct':round(100*flag.mean(),2)})
r=pd.DataFrame(out).sort_values('pct',ascending=False)
print(f'=== % ที่ถูก flag (เกณฑ์: ค่า0 ซ้ำ>={ZERO_MIN}ชม. / ค่าไม่เป็น0 ซ้ำ>={NONZERO_MIN}ชม.) ===')
print(r.to_string(index=False))
print()

total_flagged = r.flagged.sum()
total_hours = r.total_hours.sum()
pct = 100 * total_flagged / total_hours
print(f'สรุป: ข้อมูลที่ถูกติดป้ายว่าน่าสงสัย {total_flagged} ชั่วโมง จากทั้งหมด {total_hours} ชั่วโมง คิดเป็น {pct:.2f}%')

=== % ที่ถูก flag (เกณฑ์: ค่า0 ซ้ำ>=72ชม. / ค่าไม่เป็น0 ซ้ำ>=24ชม.) ===
 station_id parameter  total_hours  flagged  pct
        717       no2        38829     2510 6.46
        717       so2        47595     1445 3.04
     225611       no2         6588       98 1.49
     225611       so2        21352      220 1.03
     225611      pm25        29517      293 0.99
        717      pm25        33558      316 0.94
        717      pm10        44399      201 0.45
        717        o3        49359      182 0.37
     225611        co        12700       32 0.25
       7846       so2        20340       30 0.15
     225611      pm10        27830       30 0.11
       7846       no2        16747        0 0.00
       7846        co        17869        0 0.00
       7846      pm10        16934        0 0.00
       7846        o3        24842        0 0.00
       7846      pm25        20825        0 0.00
     225611        o3        19325        0 0.00

สรุป: ข้อมูลที่ถูกติดป้ายว่าน่าสงสัย 5357 ชั่

## สรุปผลขั้นที่ 1

- ไม่มีค่าติดลบ, ไม่มีข้อมูลซ้ำซ้อน
- ถูก flag ว่า "น่าสงสัย (เซนเซอร์ค้าง)" รวมแค่ **1.19%** ของข้อมูลทั้งหมด (5,357 จาก 448,609 ชั่วโมง)
- สารหลักของโปรเจกต์ (PM2.5, SO₂) ถูกกระทบน้อยมาก (~1-3%)
- จุดที่กระทบมากสุด (NO₂ สถานีควบคุม 6.46%) อยู่ในช่วงท้ายก่อนสถานีหยุดส่งข้อมูล ไม่ทับกับช่วงข้อมูล NO₂ ของสถานีใกล้นิคม 225611
- สถานี Field Crop Research Center (7846) สะอาดเกือบ 100% ใช้เป็นสถานีอ้างอิงตรวจทานได้

**ขั้นต่อไป (`02_cleaning.ipynb`):** ใช้เกณฑ์นี้ติดธง `is_suspect_stuck`, แปลง timezone เป็น Asia/Bangkok, pivot เป็น wide format, แล้ว merge กับข้อมูลลมจาก Open-Meteo